Libraries

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

print("Libraries imported successfully.")

Libraries imported successfully.


Data Loading

In [2]:
from google.colab import files

uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

print("File loaded successfully:", file_name)
print("Initial Dataset Shape:", df.shape)
df.head()

Saving Unemployment in India.csv to Unemployment in India.csv
File loaded successfully: Unemployment in India.csv
Initial Dataset Shape: (768, 7)


,Region,Date,Frequency,Estimated Unemployment Rate (%),Estimated Employed,Estimated Labour Participation Rate (%),Area
0,Andhra Pradesh,31-05-2019,Monthly,3.65,11999139.0,43.24,Rural
1,Andhra Pradesh,30-06-2019,Monthly,3.05,11755881.0,42.05,Rural
2,Andhra Pradesh,31-07-2019,Monthly,3.75,12086707.0,43.50,Rural
3,Andhra Pradesh,31-08-2019,Monthly,3.32,12285693.0,43.97,Rural
4,Andhra Pradesh,30-09-2019,Monthly,5.17,12256762.0,44.68,Rural


Data Checking & Cleaning

In [8]:
print("--- MISSING VALUES BEFORE CLEANING ---")
print(df.isnull().sum())

df.columns = df.columns.str.strip()

df = df.dropna(how="all")
df = df.dropna(subset=["Date", "Unemployment_Rate"])

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)

rename_dict = {
    "Region": "State",
    "Frequency": "Frequency",
    "Estimated Unemployment Rate (%)": "Unemployment_Rate",
    "Estimated Employed": "Employed",
    "Estimated Labour Participation Rate (%)": "Labour_Participation_Rate",
    "Area": "Area",
    "Region.1": "Zone",
}
df = df.rename(
    columns={k: v for k, v in rename_dict.items() if k in df.columns}
)

numeric_cols = ["Unemployment_Rate", "Employed", "Labour_Participation_Rate"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("\n--- MISSING VALUES AFTER CLEANING ---")
print(df.isnull().sum())
print("\nCleaned Dataset Shape:", df.shape)

--- MISSING VALUES BEFORE CLEANING ---
State                        0
Date                         0
Frequency                    0
Unemployment_Rate            0
Employed                     0
Labour_Participation_Rate    0
Area                         0
dtype: int64

--- MISSING VALUES AFTER CLEANING ---
State                        0
Date                         0
Frequency                    0
Unemployment_Rate            0
Employed                     0
Labour_Participation_Rate    0
Area                         0
dtype: int64

Cleaned Dataset Shape: (740, 7)


Feature Engineering

In [9]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Month_Name"] = df["Date"].dt.strftime("%b")
df["Quarter"] = df["Date"].dt.to_period("Q").astype(str)


def categorize_phase(date):
  if date < pd.Timestamp("2020-04-01"):
    return "1. Pre-Lockdown (Jan - Mar 2020 / 2019)"
  elif pd.Timestamp("2020-04-01") <= date <= pd.Timestamp("2020-05-31"):
    return "2. Peak Lockdown Shock (Apr - May 2020)"
  else:
    return "3. Unlock & Phased Recovery (Jun 2020+)"


df["Lockdown_Phase"] = df["Date"].apply(categorize_phase)

df["Unemp_to_LFPR_Ratio"] = (
    df["Unemployment_Rate"] / df["Labour_Participation_Rate"]
).round(4)

if df["Employed"].max() > 1e5:
  df["Employed_Millions"] = (df["Employed"] / 1e6).round(2)
else:
  df["Employed_Millions"] = df["Employed"]

print("Feature engineering complete. Preview of new columns:")
df[[
    "Date",
    "Lockdown_Phase",
    "Unemployment_Rate",
    "Labour_Participation_Rate",
    "Unemp_to_LFPR_Ratio",
    "Employed_Millions",
]].head()

Feature engineering complete. Preview of new columns:


,Date,Lockdown_Phase,Unemployment_Rate,Labour_Participation_Rate,Unemp_to_LFPR_Ratio,Employed_Millions
0,2019-05-31,1. Pre-Lockdown (Jan - Mar 2020 / 2019),3.65,43.24,0.0844,12.00
1,2019-06-30,1. Pre-Lockdown (Jan - Mar 2020 / 2019),3.05,42.05,0.0725,11.76
2,2019-07-31,1. Pre-Lockdown (Jan - Mar 2020 / 2019),3.75,43.50,0.0862,12.09
3,2019-08-31,1. Pre-Lockdown (Jan - Mar 2020 / 2019),3.32,43.97,0.0755,12.29
4,2019-09-30,1. Pre-Lockdown (Jan - Mar 2020 / 2019),5.17,44.68,0.1157,12.26


Summary Statistics

In [10]:
print("--- SUMMARY STATISTICS BY LOCKDOWN PHASE ---")
phase_stats = (
    df.groupby("Lockdown_Phase")
    .agg(
        Avg_Unemployment=("Unemployment_Rate", "mean"),
        Median_Unemployment=("Unemployment_Rate", "median"),
        Max_Unemployment=("Unemployment_Rate", "max"),
        Avg_LFPR=("Labour_Participation_Rate", "mean"),
        Avg_Employed_Millions=("Employed_Millions", "mean"),
    )
    .round(2)
)
display(phase_stats)

if "Area" in df.columns:
  print("\n--- URBAN VS RURAL UNEMPLOYMENT RATE BY PHASE ---")
  area_pivot = df.pivot_table(
      index="Lockdown_Phase",
      columns="Area",
      values="Unemployment_Rate",
      aggfunc="mean",
  ).round(2)
  display(area_pivot)

--- SUMMARY STATISTICS BY LOCKDOWN PHASE ---


,Avg_Unemployment,Median_Unemployment,Max_Unemployment,Avg_LFPR,Avg_Employed_Millions
Lockdown_Phase,,,,,
1. Pre-Lockdown (Jan - Mar 2020 / 2019),9.61,7.22,34.69,43.82,7.47
2. Peak Lockdown Shock (Apr - May 2020),24.26,19.96,76.74,36.82,5.58
3. Unlock & Phased Recovery (Jun 2020+),11.90,10.34,35.57,40.55,7.39



--- URBAN VS RURAL UNEMPLOYMENT RATE BY PHASE ---


Area,Rural,Urban
Lockdown_Phase,,
1. Pre-Lockdown (Jan - Mar 2020 / 2019),8.23,10.91
2. Peak Lockdown Shock (Apr - May 2020),21.48,26.93
3. Unlock & Phased Recovery (Jun 2020+),11.83,11.98


Visualization

In [13]:
national_trend = (
    df.groupby("Date")
    .agg(
        Mean_Unemployment=("Unemployment_Rate", "mean"),
        Mean_LFPR=("Labour_Participation_Rate", "mean"),
    )
    .reset_index()
)

fig1 = go.Figure()

fig1.add_trace(
    go.Scatter(
        x=national_trend["Date"],
        y=national_trend["Mean_Unemployment"],
        mode="lines+markers",
        name="Unemployment Rate (%)",
        line=dict(color="#d62728", width=3),
        marker=dict(size=6),
    )
)

fig1.add_trace(
    go.Scatter(
        x=national_trend["Date"],
        y=national_trend["Mean_LFPR"],
        mode="lines+markers",
        name="Labour Participation Rate (%)",
        line=dict(color="#1f77b4", width=2, dash="dot"),
        marker=dict(size=5),
    )
)

fig1.add_vline(
    x="2020-03-24",
    line_width=2,
    line_dash="dash",
    line_color="#2ca02c",
)

fig1.add_annotation(
    x="2020-03-24",
    y=national_trend["Mean_Unemployment"].max(),
    text="Lockdown Imposed<br>(Mar 24, 2020)",
    showarrow=True,
    arrowhead=2,
    arrowsize=1,
    arrowwidth=1.5,
    arrowcolor="#2ca02c",
    ax=-60,
    ay=-40,
    font=dict(color="#2ca02c", size=11, family="Arial Black"),
    bgcolor="rgba(255, 255, 255, 0.9)",
    bordercolor="#2ca02c",
    borderwidth=1,
    borderpad=4,
)

fig1.update_layout(
    title="<b>National Economic Shock: Unemployment Rate vs. LFPR (2019-2020)</b>",
    xaxis_title="Date",
    yaxis_title="Percentage (%)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1
    ),
    margin=dict(l=40, r=40, t=80, b=40),
)
fig1.show()

In [14]:
peak_mask = df["Lockdown_Phase"].str.contains("Peak Lockdown Shock")
state_lockdown = (
    df[peak_mask]
    .groupby("State")["Unemployment_Rate"]
    .mean()
    .reset_index()
    .sort_values(by="Unemployment_Rate", ascending=False)
    .head(10)
)

fig2 = px.bar(
    state_lockdown,
    x="Unemployment_Rate",
    y="State",
    orientation="h",
    color="Unemployment_Rate",
    color_continuous_scale="Reds",
    text=state_lockdown["Unemployment_Rate"].round(1).astype(str) + "%",
    title="<b>Top 10 Most Vulnerable States during Peak Lockdown (Apr - May 2020)</b>",
    labels={
        "Unemployment_Rate": "Avg Unemployment Rate (%)",
        "State": "State / UT",
    },
    template="plotly_white",
)

fig2.update_layout(
    yaxis=dict(autorange="reversed"),
    coloraxis_showscale=False,
    margin=dict(l=40, r=40, t=60, b=40),
)
fig2.show()

In [15]:
if "Area" in df.columns:
  fig3 = px.box(
      df,
      x="Lockdown_Phase",
      y="Unemployment_Rate",
      color="Area",
      points="outliers",
      color_discrete_map={"Rural": "#2ca02c", "Urban": "#ff7f0e"},
      title="<b>Urban vs. Rural Unemployment Volatility Across Phases</b>",
      labels={
          "Unemployment_Rate": "Unemployment Rate (%)",
          "Lockdown_Phase": "Policy Phase",
      },
      template="plotly_white",
  )
  fig3.update_layout(
      boxmode="group",
      legend=dict(
          orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1
      ),
      margin=dict(l=40, r=40, t=80, b=40),
  )
  fig3.show()
else:
  print("Area column not present in this version of the dataset.")

In [16]:
fig4 = px.scatter(
    df,
    x="Labour_Participation_Rate",
    y="Unemployment_Rate",
    color="Lockdown_Phase",
    size="Employed_Millions" if "Employed_Millions" in df.columns else None,
    hover_data=["State", "Month_Name", "Year"]
    if "Month_Name" in df.columns
    else ["State"],
    title="<b>Labor Participation vs. Unemployment Rate by Phase</b>",
    labels={
        "Labour_Participation_Rate": "Labour Participation Rate (%)",
        "Unemployment_Rate": "Unemployment Rate (%)",
    },
    template="plotly_white",
)
fig4.update_layout(margin=dict(l=40, r=40, t=60, b=40))
fig4.show()